## Multi-Scale Topological Data Analysis of Musical Structures: Phase 1 Pipeline
This notebook implements the first phase of a multi-scale topological data analysis (TDA) pipeline designed to uncover the geometric and structural invariants underlying musical compositions. Building on a carefully curated set of symbolic features—balancing rhythmic, harmonic, and textural dimensions—we deploy a novel integration of persistent homology (PH) and Mapper, where PH-derived persistence measures guide the Mapper’s lens function to reveal topologically rich clusters. This phase focuses on three scientific differentiators: (1) the PH-guided Mapper approach, which leverages local persistence as a structural descriptor; (2) the introduction of temporal and formal musical features to capture evolutionary patterns across compositions; and (3) a critical baseline comparison with classical machine learning methods, ensuring the robustness and interpretability of our topological findings. By addressing potential MIDI biases and validating our approach against both mathematical and musicological criteria, this pipeline lays the groundwork for a credible, multi-resolution analysis of musical form and style.

### 1. Imports

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import DBSCAN, KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import logging

In [4]:
# TDA libraries
from ripser import ripser
import kmapper as km
from scipy.stats import entropy

In [5]:
# Utils
import warnings
warnings.filterwarnings('ignore')

### 2. TDA Pipeline

In [5]:
"""
Topological Data Analysis Pipeline for Symbolic Music Analysis
Research Can persistent homological features of symbolic music 
simultaneously provide discriminative power for genre classification 
while revealing structural patterns that transcend genre-specific conventions?
"""

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

class MusicalTDAAnalyzer:
    """
    Core TDA analyzer for symbolic music data
    Implements a dual-perspective approach: discriminative + structural
    """
    
    def __init__(self, max_homology_dim=1, resolution=30, overlap=0.3):
        """
        Initialize TDA analyzer with optimized parameters for music data
        
        Args:
            max_homology_dim: Maximum homology dimension (0=components, 1=loops) - Phase 1
            resolution: Mapper resolution (30 = good balance speed/detail)
            overlap: Mapper overlap (0.3 = sufficient for musical continuity)
        """
        self.max_homology_dim = max_homology_dim
        self.resolution = resolution
        self.overlap = overlap
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        
        # Storage for results
        self.persistence_diagrams = {}
        self.topological_features = None
        self.mapper_graph = None
        
    def select_discriminative_features(self, df):
        """
        Feature selection based on statistical analysis findings
        Optimized 7-feature set to avoid overfitting while maintaining discriminative power
        """
        # Core discriminative features (7 total, orthogonal dimensions)
        discriminative_features = [
            'metric_weight',    # Best separability (0.68 vs 0.58)             
            'velocity',         # Natural clusters 90/100/115/125               
            'interval_to_prev', # 21% large jumps discriminate genres             
            'polyphony_notes',  # Orchestration complexity             
            'duration',         # Rhythmic signatures             
            'is_chord_tone',    # Harmonic anchoring             
            'local_key'         # Harmonic sophistication
        ]
        
        return df[discriminative_features + ['genre']].copy()
    
    def create_structural_representation(self, df):
        """
        Creates 7-dimensional structural representation for persistent homology
        Optimized feature space avoiding redundancy
        """
        # Remove genre for structural analysis
        structural_data = df.drop('genre', axis=1)
        
        # Normalize features for TDA
        normalized_data = self.scaler.fit_transform(structural_data)
        
        # Return normalized 7D feature space directly (no composite features)
        # Each dimension represents one orthogonal musical aspect
        return normalized_data
    
    def _compute_optimal_min_points(self, X, genre_labels):
        """
        Computes optimal minimum points per genre for meaningful TDA
        Based on intrinsic dimensionality and statistical requirements
        """
        unique_genres = np.unique(genre_labels)
        genre_sizes = []
        
        for genre in unique_genres:
            genre_mask = genre_labels == genre
            genre_size = np.sum(genre_mask)
            genre_sizes.append(genre_size)
        
        # Statistical requirements for persistent homology:
        # - Need enough points to form simplices: min = feature_dim + 2
        # - Need statistical significance: min = 50 points for stable results
        # - Adaptive based on smallest genre: use 10% of smallest genre or 50, whichever is larger
        
        min_genre_size = min(genre_sizes)
        feature_dim = X.shape[1]
        
        statistical_min = max(50, feature_dim + 2)  # Theoretical minimum
        adaptive_min = max(min_genre_size // 10, statistical_min)  # 10% of smallest genre
        
        optimal_min = min(adaptive_min, min_genre_size // 2)  # Don't exceed 50% of smallest genre
        
        print(f"Genre sizes range: {min(genre_sizes)} - {max(genre_sizes)}")
        print(f"Optimal minimum points per genre: {optimal_min}")
        
        return optimal_min
    
    def compute_persistent_homology(self, X, genre_labels):
        """
        Computes persistent homology for each genre and globally
        Phase 1: Robust approach with optimal thresholds and error handling
        """
        logger.info("Computing persistent homology...")
        
        # Compute optimal minimum points threshold
        min_points = self._compute_optimal_min_points(X, genre_labels)
        successful_genres = 0
        failed_genres = []

        # Genre-specific persistence diagrams
        unique_genres = np.unique(genre_labels)
        
        for genre in unique_genres:
            genre_mask = genre_labels == genre
            genre_data = X[genre_mask]
            
            if len(genre_data) >= min_points:
                try:
                    genre_persistence = ripser(genre_data, maxdim=self.max_homology_dim)
                    
                    # Verify non-empty result and valid structure
                    if (genre_persistence and 'dgms' in genre_persistence and 
                        len(genre_persistence['dgms']) > 0 and
                        len(genre_persistence['dgms'][0]) > 0):  # At least H0 exists
                        
                        self.persistence_diagrams[genre] = genre_persistence
                        successful_genres += 1
                    else:
                        failed_genres.append(f"{genre} (empty diagram)")
                        logger.warning(f"Empty persistence diagram for genre {genre}")
                        
                except Exception as e:
                    failed_genres.append(f"{genre} (error: {str(e)[:50]})")
                    logger.error(f"Error computing persistence for genre {genre}: {e}")
            else:
                failed_genres.append(f"{genre} (insufficient points: {len(genre_data)})")
                logger.warning(f"Skipping genre {genre}: insufficient points ({len(genre_data)} < {min_points})")
        
        if successful_genres < 3:
            logger.critical(f"⚠ CRITICAL: Only {successful_genres} genres successfully processed")
            logger.error(f"Failed genres: {failed_genres}")
            logger.critical("⚠ Insufficient data for meaningful topological analysis")
            logger.warning("⚠ Consider reducing min_points threshold or increasing corpus size")
        
            # Fallback
            reduced_min_points = max(10, min_points // 2)
            logger.info(f"Attempting fallback with reduced threshold: {reduced_min_points}")
        
            for genre in unique_genres:
                if genre not in self.persistence_diagrams:
                    genre_mask = genre_labels == genre
                    genre_data = X[genre_mask]
                    
                    if len(genre_data) >= reduced_min_points:
                        try:
                            genre_persistence = ripser(genre_data, maxdim=0)  # H0 seulement
                            if (genre_persistence and 'dgms' in genre_persistence and 
                                len(genre_persistence['dgms']) > 0):
                                self.persistence_diagrams[genre] = genre_persistence
                                successful_genres += 1
                                print(f"✓ Fallback success for {genre}")
                        except:
                            pass

        print(f"Successfully computed persistence for {successful_genres}/{len(unique_genres)} genres")
        print(f"Total structures: {len(self.persistence_diagrams)}")
        
        return self.persistence_diagrams
    
    def extract_topological_features(self, persistence_diagrams):
        """
        Extracts discriminative features from persistence diagrams
        Returns structured matrix (n_genres × n_features_per_genre) for classification
        """
        # Filter out global diagram and organize by genre
        genre_diagrams = {k: v for k, v in persistence_diagrams.items() if k != 'global'}
        
        if not genre_diagrams:
            logger.warning("No genre-specific persistence diagrams available")
            return np.array([]), [], []
        
        # Sort genres for consistent ordering
        sorted_genres = sorted(genre_diagrams.keys())
        n_features_per_genre = 6  # 3 H0 + 3 H1 features per genre
        
        features_matrix = []
        feature_names = []
        valid_genres = []
        
        for genre in sorted_genres:
            diagram = genre_diagrams[genre]
            genre_features = []
            
            # Verify diagram structure
            if not diagram or 'dgms' not in diagram:
                logger.warning(f"Invalid diagram structure for {genre}, skipping")
                continue
                
            # H0 features (connected components)
            if len(diagram['dgms']) > 0 and len(diagram['dgms'][0]) > 0:
                h0_points = diagram['dgms'][0]
                h0_lifetimes = h0_points[:, 1] - h0_points[:, 0]
                h0_lifetimes = h0_lifetimes[np.isfinite(h0_lifetimes)]
                
                genre_features.extend([
                    len(h0_points),
                    np.mean(h0_lifetimes) if len(h0_lifetimes) > 0 else 0,
                    np.std(h0_lifetimes) if len(h0_lifetimes) > 0 else 0
                ])
            else:
                genre_features.extend([0, 0, 0])
            
            # H1 features (loops) with robust verification
            if (len(diagram['dgms']) > 1 and 
                diagram['dgms'][1] is not None and 
                len(diagram['dgms'][1]) > 0):
                
                h1_points = diagram['dgms'][1]
                valid_h1_mask = np.isfinite(h1_points[:, 0]) & np.isfinite(h1_points[:, 1])
                h1_points_valid = h1_points[valid_h1_mask]
                
                if len(h1_points_valid) > 0:
                    h1_lifetimes = h1_points_valid[:, 1] - h1_points_valid[:, 0]
                    h1_lifetimes = h1_lifetimes[h1_lifetimes > 0]
                    
                    genre_features.extend([
                        len(h1_points_valid),
                        np.mean(h1_lifetimes) if len(h1_lifetimes) > 0 else 0,
                        np.max(h1_lifetimes) if len(h1_lifetimes) > 0 else 0
                    ])
                else:
                    genre_features.extend([0, 0, 0])
            else:
                genre_features.extend([0, 0, 0])
            
            # Add to results if we have the expected number of features
            if len(genre_features) == n_features_per_genre:
                features_matrix.append(genre_features)
                valid_genres.append(genre)
            else:
                logger.warning(f"Unexpected feature count for {genre}: {len(genre_features)}")
        
        # Create feature names (same structure for all genres)
        if valid_genres:
            base_names = ['h0_count', 'h0_mean_life', 'h0_std_life', 
                         'h1_count', 'h1_mean_life', 'h1_max_life']
            feature_names = [f"topo_{name}" for name in base_names]
        
        features_array = np.array(features_matrix) if features_matrix else np.array([])
        
        print(f"Extracted topological features for {len(valid_genres)} genres")
        if len(features_array) > 0:
            print(f"Feature matrix shape: {features_array.shape}")
        
        return features_array, feature_names, valid_genres
    
    def build_mapper_graph(self, processed_data, genre_labels): 
        """
        Builds Mapper graph for structural pattern discovery
        Reveals trans-genre organizational principles
        """
        logger.info("Building Mapper graph...")
        
        # Initialize Mapper
        mapper = km.KeplerMapper(verbose=1)
        
        # Create lens function based on key discriminative features
        # Focus on rhythmic + harmonic dimensions (preserves musical interpretability)
        print("  Using musical features lens (preserves semantic information)") 

        X = processed_data.drop('genre', axis=1).values
        X = self.scaler.fit_transform(X)  # Normalize here
        
        # Safe feature indexing using actual column names
        feature_names = list(processed_data.drop('genre', axis=1).columns)
        lens_features = ['metric_weight', 'velocity', 'is_chord_tone', 'local_key']
        
        # Find available and missing features
        available_features = [f for f in lens_features if f in feature_names]
        missing_features = [f for f in lens_features if f not in feature_names]
        
        if missing_features:
            logger.warning(f"Missing lens features: {missing_features}")
        
        if len(available_features) >= 2:  # Minimum for meaningful lens
            lens_indices = [feature_names.index(f) for f in available_features]
            lens = X[:, lens_indices]
            print(f"Using {len(available_features)} available features: {available_features}")
        else:
            logger.error(f"Critical: Only {len(available_features)} lens features available, cannot build meaningful lens")
            raise ValueError(f"Insufficient features for lens construction: only {available_features} available")
    
        # Build Mapper graph
        self.mapper_graph = mapper.map(
            lens=lens,
            X=X,
            clusterer=DBSCAN(eps=0.3, min_samples=3),
            cover=km.Cover(n_cubes=self.resolution, perc_overlap=self.overlap)
        )
        
        return self.mapper_graph
    
    def compute_structural_metrics(self, mapper_graph, genre_labels):
        """
        Computes comprehensive metrics for structural pattern analysis
        Addresses the trans-genre structure question with multiple indicators
        """
        if not mapper_graph or 'nodes' not in mapper_graph:
            return {}
        
        metrics = {}
        
        # Basic graph topology metrics
        n_nodes = len(mapper_graph['nodes'])
        # Count actual edges from adjacency structure
        # Note: Assumes KeplerMapper returns links as dict format {node: [neighbors]}
        # Phase 2: Add support for edge list format [(node1, node2), ...] if needed
        n_edges = sum(len(neighbors) for neighbors in mapper_graph['links'].values()) // 2  # Divide by 2 for undirected
        
        metrics['graph_density'] = (2 * n_edges / (n_nodes * (n_nodes - 1))) if n_nodes > 1 else 0.0
        metrics['n_components'] = self._count_connected_components(mapper_graph)
        metrics['n_nodes'] = n_nodes
        metrics['n_edges'] = n_edges
        
        # Node size analysis (cluster sizes)
        node_sizes = [len(node_members) for node_members in mapper_graph['nodes'].values()]
        if node_sizes:
            metrics['avg_cluster_size'] = np.mean(node_sizes)
            metrics['cluster_size_std'] = np.std(node_sizes)
            metrics['max_cluster_size'] = np.max(node_sizes)
            metrics['min_cluster_size'] = np.min(node_sizes)
        
        # Genre mixing analysis (trans-genre patterns)
        node_genre_diversity = []
        node_genre_counts = []
        
        # Get total number of unique genres for normalization
        unique_genres_count = len(np.unique(genre_labels))
        max_entropy = np.log(unique_genres_count) if unique_genres_count > 1 else 1.0
        
        for node_id, node_members in mapper_graph['nodes'].items():
            node_genres = [genre_labels[int(member)] for member in node_members if int(member) < len(genre_labels)]
            if node_genres:
                genre_counts = pd.Series(node_genres).value_counts(normalize=True)
                raw_entropy = entropy(genre_counts.values)  # Shannon entropy
                normalized_diversity = raw_entropy / max_entropy  # Normalize to [0,1]
                node_genre_diversity.append(normalized_diversity)
                node_genre_counts.append(len(genre_counts))  # Number of distinct genres in node
        
        if node_genre_diversity:
            metrics['avg_genre_diversity'] = np.mean(node_genre_diversity)
            metrics['max_genre_diversity'] = np.max(node_genre_diversity)
            metrics['std_genre_diversity'] = np.std(node_genre_diversity)
            
            # Genre mixing intensity
            metrics['avg_genres_per_node'] = np.mean(node_genre_counts)
            metrics['max_genres_per_node'] = np.max(node_genre_counts)
            
            # Structural cohesion vs diversity
            high_diversity_nodes = np.sum(np.array(node_genre_diversity) > 0.7) # Use normalized entropy threshold (0.7 = 70% of maximum diversity)
            metrics['high_diversity_nodes'] = high_diversity_nodes
            metrics['high_diversity_ratio'] = high_diversity_nodes / n_nodes if n_nodes > 0 else 0
        
        return metrics
    
    def _count_connected_components(self, graph):
        """
        Helper function to count connected components in Mapper graph
        Note: Assumes KeplerMapper links format {node_id: [neighbor_ids]}
        """
        visited = set()
        components = 0
        
        def dfs(node):
            visited.add(node)
            # Assumes dict format for links structure
            for neighbor in graph.get('links', {}).get(node, []):
                if neighbor not in visited:
                    dfs(neighbor)
        
        for node in graph['nodes']:
            if node not in visited:
                dfs(node)
                components += 1
        
        return components

class MusicalTDAPipeline:
    """
    Complete pipeline implementing the dual-perspective TDA approach
    Phase 1: Proof of concept with solid foundation for extension
    """
    
    def __init__(self):
        self.analyzer = MusicalTDAAnalyzer()
        self.baseline_classifier = RandomForestClassifier(
            n_estimators=100, 
            random_state=42,
            class_weight='balanced'
        )
        
        # Results storage
        self.results = {
            'discriminative_power': {},
            'structural_patterns': {},
            'baseline_performance': {}
        }

    def _evaluate_fusion_performance(self, X_fused, y_fused, feature_names):
        """Evaluate fused features performance with robust validation"""
        if len(np.unique(y_fused)) < 2 or len(X_fused) < 3:
            print("Skipping fusion evaluation: insufficient data for meaningful CV")
            return
        
        # Check class distribution
        counts = np.unique(y_fused, return_counts=True)
        min_class_size = min(counts)
        if min_class_size < 2:
            print(f"Skipping fusion evaluation: smallest class has {min_class_size} samples")
            return
        
        cv_folds = min(3, min_class_size, len(X_fused))
        if cv_folds < 2:
            print(f"Skipping fusion evaluation: insufficient folds ({cv_folds})")
            return
        
        cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
        fused_scores = cross_val_score(self.baseline_classifier, X_fused, y_fused, cv=cv)
        
        self.results['discriminative_power']['fused_accuracy'] = {
            'mean': fused_scores.mean(),
            'std': fused_scores.std(),
            'n_genres': len(X_fused),
            'n_features': len(feature_names)
        }
        print(f"Fused Accuracy: {fused_scores.mean():.3f} (+/- {fused_scores.std()*2:.3f})")
    
    def run_analysis(self, df):
        """
        Main pipeline execution
        Answers: Can persistent homology provide both discriminative power AND reveal trans-genre patterns?
        """
        print("=" * 60)
        print("MUSICAL TDA PIPELINE - PHASE 1")
        print("=" * 60)
        
        # 1. Feature Selection and Preparation
        print("\n1. FEATURE SELECTION...")
        processed_data = self.analyzer.select_discriminative_features(df)
        print(f"Selected {len(processed_data.columns)-1} discriminative features")
        print("Features:", list(processed_data.columns[:-1]))
        
        # 2. Create Structural Representation
        print("\n2. STRUCTURAL REPRESENTATION...")
        X = self.analyzer.create_structural_representation(processed_data)
        y = self.analyzer.label_encoder.fit_transform(processed_data['genre'])
        genre_labels = processed_data['genre'].values
        
        print(f"Created {X.shape[1]}D structural representation for {X.shape[0]} musical events")
        
        # 3. Persistent Homology Analysis
        print("\n3. PERSISTENT HOMOLOGY...")
        persistence_diagrams = self.analyzer.compute_persistent_homology(X, genre_labels)
        
        # 4. Topological Feature Extraction
        print("\n4. TOPOLOGICAL FEATURE EXTRACTION...")
        topo_features, feature_names, valid_genres = self.analyzer.extract_topological_features(persistence_diagrams)
        self.analyzer.topological_features = topo_features
        self.analyzer.valid_genres = valid_genres
        
        if len(topo_features) > 0:
            print(f"Extracted features for {len(valid_genres)} valid genres")
            print("Valid genres:", valid_genres[:5] if len(valid_genres) > 5 else valid_genres)
            print("Sample features:", feature_names[:3])
        else:
            logger.warning("No valid topological features extracted")
        # 5. Mapper Analysis for Structural Patterns
        print("\n5. MAPPER GRAPH CONSTRUCTION...")
        mapper_graph = self.analyzer.build_mapper_graph(processed_data, genre_labels)
        
        # 6. Structural Pattern Analysis
        print("\n6. STRUCTURAL PATTERN ANALYSIS...")
        structural_metrics = self.analyzer.compute_structural_metrics(mapper_graph, genre_labels)
        self.results['structural_patterns'] = structural_metrics
        
        print("Structural Metrics:")
        for metric, value in structural_metrics.items():
            print(f"  {metric}: {value:.4f}")
        
        # 7. Discriminative Power Evaluation
        print("\n7. DISCRIMINATIVE POWER EVALUATION...")
        X_baseline = processed_data.drop('genre', axis=1)
        self._evaluate_discriminative_power(X_baseline, y)

        # 8. Fusion Analysis (Phase 1 validation)
        print("\n8. FUSION ANALYSIS...")
        X_fused, y_fused, fused_names = self.create_fused_genre_dataset(
            processed_data, topo_features, valid_genres, feature_names
        )
        if X_fused is not None:
            self._evaluate_fusion_performance(X_fused, y_fused, fused_names)
        
        # 9. Results Summary
        print("\n9. ANALYSIS COMPLETE!")
        self._print_results_summary()
        
        return self.results
    
    def aggregate_original_features_by_genre(self, df, agg_funcs=None):
        """
        Aggregates original event-level features per genre (mean, median) to match
        the genre-level topological features granularity.
        Returns DataFrame indexed by genre.
        """
        if agg_funcs is None:
            agg_funcs = ['mean', 'median']
        # keep only numeric feature columns (exclude 'genre')
        numeric_cols = [c for c in df.columns if c != 'genre' and np.issubdtype(df[c].dtype, np.number)]
        if 'genre' in numeric_cols:
            numeric_cols.remove('genre')
        agg_df = df.groupby('genre')[numeric_cols].agg(agg_funcs)
        # flatten columns: ('duration','mean') -> 'duration_mean'
        agg_df.columns = ['_'.join(col).strip() for col in agg_df.columns.values]
        agg_df = agg_df.reset_index().set_index('genre')
        return agg_df
    
    def create_fused_genre_dataset(self, processed_data, topo_features, valid_genres, topo_feature_names):
        """
        Build a fused dataset at genre-level:
        - aggregate original features per genre (mean/median)
        - align genres with topo_features (valid_genres)
        - concat aggregated originals + topo features
        Returns X_fused (ndarray), y_fused (labels), fused_feature_names (list)
        """
        # 1) aggregate originals by genre
        agg_orig = self.aggregate_original_features_by_genre(processed_data)  # DataFrame indexed by genre
        
        # 2) prepare topo DataFrame (rows per genre in same order as valid_genres)
        if topo_features is None or len(topo_features) == 0:
            logger.warning("No topological features available for fusion")
            return None, None, None
        if not valid_genres or len(valid_genres) == 0:
            logger.warning("No valid genres available for fusion")
            return None, None, None
        
        # 3) prepare topo DataFrame
        topo_df = pd.DataFrame(topo_features, index=valid_genres, columns=topo_feature_names)
        
        # 4) intersect genres present in both
        common_genres = [g for g in valid_genres if g in agg_orig.index]
        if len(common_genres) == 0:
            print("CRITICAL: No overlapping genres between original aggregates and topo features")
            print(f"Valid topo genres: {valid_genres[:5]}...")
            print(f"Available orig genres: {list(agg_orig.index)[:5]}...")
            return None, None, None
        
        # 5) verify minimal size for classification
        if len(common_genres) < 3:
            logger.warning(f"Only {len(common_genres)} common genres available, insufficient for robust classification")

        # 6) verify all common genres exist in both datasets before indexing
        try:
            agg_sub = agg_orig.loc[common_genres]
            topo_sub = topo_df.loc[common_genres]
        except KeyError as e:
            print(f"ERROR: Genre indexing failed - {e}")
            print(f"Missing from agg_orig: {set(common_genres) - set(agg_orig.index)}")
            print(f"Missing from topo_df: {set(common_genres) - set(topo_df.index)}")
            return None, None, None

        # 7) concat (axis=1) and return
        try:
            fused_df = pd.concat([agg_sub, topo_sub], axis=1, join="inner")  # Index-based join
            fused_feature_names = list(fused_df.columns)
            X_fused = fused_df.values
            
            label_encoder = LabelEncoder()
            y_fused = label_encoder.fit_transform(common_genres)
            
            print(f"Fusion successful: {len(common_genres)} genres, {X_fused.shape[1]} features")
            return X_fused, y_fused, fused_feature_names
            
        except Exception as e:
            print(f"Error during fusion: {e}")
            return None, None, None
    

    def evaluate_topological_clustering(self, topo_features, genre_labels):
        """
        Evaluate topological features via unsupervised clustering metrics 
        More appropriate than classification at genre-aggregated level.
        """
        if len(topo_features) < 3:
            logger.warning("Skipping clustering evaluation: insufficient topo features")
            return None
        
        # Convert genre_labels to numeric if needed
        genre_arr = np.array(genre_labels)
        if not np.issubdtype(genre_arr.dtype, np.number):
            genre_labels = self.analyzer.label_encoder.fit_transform(genre_labels)
        
        # Number of clusters = bounded by unique genres and number of samples
        n_unique_genres = len(np.unique(genre_labels))
        n_clusters = min(n_unique_genres, len(topo_features))
        if n_clusters < 2:
            logger.warning("Skipping clustering evaluation: only one cluster possible")
            return None
        
        # KMeans clustering on topological features
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(topo_features)
        
        # Compare clusters to genre labels
        ari_score = adjusted_rand_score(genre_labels, cluster_labels)
        nmi_score = normalized_mutual_info_score(genre_labels, cluster_labels)
        
        print(f"Clustering evaluation: ARI={ari_score:.3f}, NMI={nmi_score:.3f} ({n_clusters} clusters)")
        
        return {
            'clustering_ari': ari_score,
            'clustering_nmi': nmi_score,
            'n_clusters': n_clusters,
            'n_unique_genres': n_unique_genres
        }

    def _evaluate_discriminative_power(self, X_original, y):
        """
        Evaluates genre classification performance
        Compares original features vs topological features with proper synchronization
        """
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        # Baseline performance (original features)
        baseline_scores = cross_val_score(
            self.baseline_classifier, 
            X_original, 
            y, 
            cv=cv, 
            scoring='accuracy'
        )
        
        self.n_original_samples = len(X_original)
        if hasattr(self.analyzer, 'valid_genres'):
            self.n_valid_genres = len(self.analyzer.valid_genres)
        else:
            self.n_valid_genres = None

        self.results['discriminative_power']['baseline_accuracy'] = {
            'mean': baseline_scores.mean(),
            'std': baseline_scores.std()
        }
        
        print(f"Baseline Accuracy: {baseline_scores.mean():.3f} (+/- {baseline_scores.std()*2:.3f})")
        
        # Topological clustering evaluation (replaces classification for genre-level)
        if (hasattr(self.analyzer, 'topological_features') and 
            self.analyzer.topological_features is not None and 
            len(self.analyzer.topological_features) > 0 and
            hasattr(self.analyzer, 'valid_genres')):
            
            topo_features = self.analyzer.topological_features
            valid_genres = self.analyzer.valid_genres
            
            clustering_results = self.evaluate_topological_clustering(topo_features, valid_genres)
            self.results['discriminative_power']['topological_clustering'] = clustering_results
            
            if clustering_results:
                print(f"Topological Clustering Evaluation:")
                print(f"  ARI: {clustering_results['clustering_ari']:.3f}")
                print(f"  NMI: {clustering_results['clustering_nmi']:.3f}")
                print(f"  Clusters: {clustering_results['n_clusters']}")
            else:
                print("Topological clustering evaluation failed")
        else:
            print("No valid topological features available for clustering evaluation")
            self.results['discriminative_power']['topological_clustering'] = None
    
    def _print_results_summary(self):
        """
        Prints comprehensive results summary
        Addresses the research question directly
        """
        print("\n" + "="*60)
        print("RESEARCH QUESTION RESULTS")
        print("="*60)
        
        print("\nQUESTION: Can persistent homology provide discriminative power")
        print("while revealing trans-genre structural patterns?")
        
        print("\nDISCRIMINATIVE POWER:")
        baseline_acc = self.results['discriminative_power'].get('baseline_accuracy', {})
        topo_clustering = self.results['discriminative_power'].get('topological_clustering')
        print(f"  Baseline Classification: {baseline_acc.get('mean', 0):.3f}")
        
        if topo_clustering:
            print(f"  Topological Clustering: ARI={topo_clustering.get('clustering_ari', 0):.3f}, NMI={topo_clustering.get('clustering_nmi', 0):.3f}")
            print(f"    (n_genres={topo_clustering.get('n_unique_genres', 'N/A')})")
        
        print("\nSTRUCTURAL PATTERNS:")
        structural = self.results['structural_patterns']
        if structural:
            # Core connectivity
            components = structural.get('n_components', 0)
            density = structural.get('graph_density', 0)
            n_nodes = structural.get('n_nodes', 0)
            
            print(f"  Graph Structure: {n_nodes} nodes, {components} components, density={density:.3f}")
            
            # Cluster characteristics
            avg_size = structural.get('avg_cluster_size', 0)
            size_std = structural.get('cluster_size_std', 0)
            print(f"  Cluster Sizes: μ={avg_size:.1f}, σ={size_std:.1f}")
            
            # Genre mixing analysis
            diversity = structural.get('avg_genre_diversity', 0)
            max_diversity = structural.get('max_genre_diversity', 0)
            high_div_ratio = structural.get('high_diversity_ratio', 0)
            
            print(f"  Genre Mixing: avg_entropy={diversity:.3f}, max_entropy={max_diversity:.3f}")
            print(f"  High-diversity nodes: {high_div_ratio:.1%}")
            
            # Interpretation
            if high_div_ratio > 0.3:  # >30% nodes with high genre diversity
                print("  → STRONG EVIDENCE of trans-genre structural patterns")
            elif diversity > 0.5:  # Moderate mixing
                print("  → MODERATE EVIDENCE of trans-genre structural patterns")  
            else:
                print("  → LIMITED trans-genre patterns detected")
        
        # Performance comparison with limitations disclaimer
        print(f"\nDISCRIMINATIVE POWER ANALYSIS:")
        print(f"⚠️  METHODOLOGICAL LIMITATIONS:")
        print(f"  - Baseline: Event-level features ({getattr(self, 'n_original_samples', 'N/A')} samples)")
        print(f"  - Topological: Genre-level features (~{getattr(self, 'n_valid_genres', 'N/A')} samples)")
        print(f"  - Granularity mismatch limits direct comparison validity")
        print(f"  - Results are indicative for Phase 1 proof-of-concept only")
        print(f"  Baseline Classification: {baseline_acc.get('mean', 0):.3f} ± {baseline_acc.get('std', 0):.3f}")
    
        if topo_clustering:
            print(f"  Topological Features: ARI={topo_clustering.get('clustering_ari', 0):.3f}, NMI={topo_clustering.get('clustering_nmi', 0):.3f}")
            print(f"    (n_genres={topo_clustering.get('n_unique_genres', 'N/A')})")
        
        fused_acc = self.results['discriminative_power'].get('fused_accuracy')
        if fused_acc:
            print(f"  Fused Features: {fused_acc.get('mean', 0):.3f} ± {fused_acc.get('std', 0):.3f}")
            print(f"    (n_genres={fused_acc.get('n_unique_genres', 'N/A')}, n_features={fused_acc.get('n_features', 'N/A')})")
            
            # Complementarity assessment
            baseline_mean = baseline_acc.get('mean', 0)
            fused_mean = fused_acc.get('mean', 0) 
            if fused_mean > baseline_mean + 0.05:  # 5% improvement threshold
                print("  → POTENTIAL COMPLEMENTARITY: Fusion shows improvement")
            elif abs(fused_mean - baseline_mean) < 0.05:
                print("  → NEUTRAL: No clear complementarity detected")
            else:
                print("  → LIMITED COMPLEMENTARITY: Fusion underperforms")
        
        # Overall conclusion
        structural_evidence = high_div_ratio > 0.3 if structural and 'high_diversity_ratio' in structural else False
        discriminative_evidence = topo_clustering and topo_clustering.get('clustering_ari', 0) > 0.3  # ARI > 0.3 = moderate agreement
        
        if structural_evidence and discriminative_evidence:
            conclusion = "Phase 1 successfully demonstrates TDA potential for musical analysis!"
        elif structural_evidence or discriminative_evidence:  
            conclusion = "Phase 1 provides partial validation, Phase 2 will strengthen evidence."
        else:
            conclusion = "Phase 1 establishes methodological foundation for Phase 2 development."

        print(f"\n{'='*60}")
        print("FINAL CONCLUSION:")
        print(f"{'='*60}")
        print(f"  {conclusion}")

def main():
    """
    Usage with processed data
    """
    print("TDA Musical Analysis Pipeline - Processed Data")
    
    # Import features
    required_cols = [
        'metric_weight', 'velocity', 'duration', 
        'interval_to_prev', 'polyphony_notes', 
        'is_chord_tone', 'local_key', 'genre'
    ]
    df = pd.read_parquet('datasets/processed_midi_data.parquet')[required_cols]
    
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in dataframe: {missing}")
    
    print(f"Original dataset: {df.shape}")
    if len(df) > 10000:  # Adaptive threshold
        df = (
            df.groupby('genre')
              .apply(lambda x: x.sample(n=min(len(x), 1000), random_state=42))
              .reset_index(drop=True)
        )
        print(f"Subsampled to: {df.shape} (max 1000 samples per genre)")
    
    # Run pipeline
    pipeline = MusicalTDAPipeline()
    results = pipeline.run_analysis(df)

    logger.info(f"Results (keys): {list(results.keys())}")
    logger.info(f"Discriminative power: {results.get('discriminative_power', {})}")
    
    return results

if __name__ == "__main__":
    main()

TDA Musical Analysis Pipeline - Processed Data


INFO: Computing persistent homology...


MUSICAL TDA PIPELINE - PHASE 1

1. FEATURE SELECTION...
Selected 7 discriminative features
Features: ['metric_weight', 'velocity', 'interval_to_prev', 'polyphony_notes', 'duration', 'is_chord_tone', 'local_key']

2. STRUCTURAL REPRESENTATION...
Created 7D structural representation for 638769 musical events

3. PERSISTENT HOMOLOGY...
Genre sizes range: 28830 - 54471
Optimal minimum points per genre: 2883


: 

: 

---

In [6]:
required_cols = ['metric_weight', 'velocity', 'duration', 
                     'interval_to_prev', 'polyphony_notes', 
                     'is_chord_tone', 'local_key', 'genre']
df = pd.read_parquet('datasets/processed_midi_data.parquet')[required_cols]
df

,metric_weight,velocity,duration,interval_to_prev,polyphony_notes,is_chord_tone,local_key,genre
0,1.0,-1,4.00,0,0,0,3,pop
1,1.0,-1,4.00,0,0,0,3,pop
2,1.0,-1,4.00,0,0,0,3,pop
3,1.0,-1,4.00,0,0,0,3,pop
4,1.0,-1,4.00,0,0,0,3,pop
...,...,...,...,...,...,...,...,...
638764,1.0,50,1.75,7,5,1,16,jazz
638765,1.0,51,2.00,4,5,1,16,jazz
638766,0.6,46,1.25,31,2,0,16,jazz
638767,0.6,46,1.25,0,2,0,16,jazz


In [7]:
df.shape

(638769, 8)